In [2]:
import torch
import pandas as pd

# 重み（2行×3列）
W = torch.tensor([[0.9, -0.6, -0.9],
                  [0.6, -0.6,  0.3]])
# 入力（4行×3列）
X = torch.tensor([[1., 2., 1.],
                  [2., 1., 2.],
                  [0., 1., 0.],
                  [1., 1., 1.]])

# ビットは2で固定(0,1,2,3)
bits = 2

In [3]:
def make_grid(W, bits = 2):
  maxq = 2**bits - 1                                     # ex) bits = 2 --> maxq = 3
  xmin = W.min(dim=1, keepdim=True).values.clamp(max=0)
  xmax = W.max(dim=1, keepdim=True).values.clamp(min=0)
  scale = (xmax - xmin) / maxq
  zero = torch.round(-xmin / scale)
  print(f"xmin: {xmin}\nxmax: {xmax}\nscale: {scale}\nzero: {zero}")
  return scale, zero, maxq

# 丸める関数
def quant(x, scale, zero, maxq, is_print_values=False):

  def idx_print(title):
    "表示用関数"
    if is_print_values == False: return
    print(f"{title}: {idx}")

  idx = x / scale                  ; idx_print("idx = x / scale")
  idx = torch.round(idx)           ; idx_print("torch.round(idx)")
  idx += zero                      ; idx_print("idx + zero")
  idx = torch.clamp(idx, 0, maxq)  ; idx_print("torch.clamp(idx, 0, maxq)")
  return scale * (idx - zero)

In [4]:
scale, zero, maxq = make_grid(W, bits)
s, z = scale[0, 0], zero[0, 0]       # 1行目の目盛り

w0 = W[0]

# RTN (1行目だけ)
w1 = quant(W[0], s, z, maxq, True)

xmin: tensor([[-0.9000],
        [-0.6000]])
xmax: tensor([[0.9000],
        [0.6000]])
scale: tensor([[0.6000],
        [0.4000]])
zero: tensor([[2.],
        [2.]])
idx = x / scale: tensor([ 1.5000, -1.0000, -1.5000])
torch.round(idx): tensor([ 2., -1., -2.])
idx + zero: tensor([4., 1., 0.])
torch.clamp(idx, 0, maxq): tensor([3., 1., 0.])


In [5]:
# GPTQ 前準備
H = 2 * X.T @ X / 4                                         # 入力の連動の表
H = H + 0.01 * H.diag().mean() * torch.eye(3)               # ダンピング
U = torch.linalg.cholesky(torch.linalg.inv(H), upper=True)
ratio = U / U.diag()[:, None]                               # 比率[j][k] = U[j,k] / U[j,j]

# 表示用
cols = ["x1", "x2", "x3"]

df_H = pd.DataFrame(H.numpy(), index=cols, columns=cols).round(3)
df_ratio = pd.DataFrame(ratio.numpy(), index=cols, columns=cols).round(3)

print("入力の連動の表(ヘッセ行列 H_F)")
display(df_H)
print("\n比率の表(ずれたら、どの列をどれだけ動かすか)")
display(df_ratio)

入力の連動の表(ヘッセ行列 H_F)


,x1,x2,x3
x1,3.032,2.500,3.000
x2,2.500,3.532,2.500
x3,3.000,2.500,3.032



比率の表(ずれたら、どの列をどれだけ動かすか)


,x1,x2,x3
x1,1.0,-0.018,-0.975
x2,0.0,1.000,-0.825
x3,0.0,0.000,1.000


In [6]:
# GPTQ (1行目だけ)
w2 = w0.clone()                 # 1行目 [0.9, -0.6, -0.9] をコピー
for j in range(3):
  print(f"=== {j}列目 ======================================================")
  q = quant(w2[j], s, z, maxq)  # 1. 丸める

  d = w2[j] - q                 # 2. ずれ

  w2[j] = q
  print(f"丸めた後={w2.tolist()}\nずれ={d.item():.3f}")

  for k in range(j + 1, 3):    # 3. 右側を動かす
      w2[k] -= d * ratio[j, k]

  print(f"ずれ修正後の重み={w2.tolist()}")

# ---- RTNと比べる ----
print("初期値:", W[0].tolist())
print("RTN   :", w1.tolist())
print("GPTQ  :", w2.tolist())

=== 0列目 ======================================================
丸めた後=[0.5999999642372131, -0.6000000238418579, -0.8999999761581421]
ずれ=0.300
ずれ修正後の重み=[0.5999999642372131, -0.5946711301803589, -0.6075279712677002]
=== 1列目 ======================================================
丸めた後=[0.5999999642372131, -0.5999999642372131, -0.6075279712677002]
ずれ=0.005
ずれ修正後の重み=[0.5999999642372131, -0.5999999642372131, -0.6031336784362793]
=== 2列目 ======================================================
丸めた後=[0.5999999642372131, -0.5999999642372131, -0.5999999642372131]
ずれ=-0.003
ずれ修正後の重み=[0.5999999642372131, -0.5999999642372131, -0.5999999642372131]
初期値: [0.8999999761581421, -0.6000000238418579, -0.8999999761581421]
RTN   : [0.5999999642372131, -0.5999999642372131, -1.1999999284744263]
GPTQ  : [0.5999999642372131, -0.5999999642372131, -0.5999999642372131]


In [7]:
pd.set_option("display.float_format", "{:.6f}".format)   # 小数点以下6桁で表示
weights = pd.DataFrame({"元": w0.numpy(), "RTN": w1.numpy(), "GPTQ": w2.numpy()},
                       index=cols).round(3)
display(weights)

# ---- 出力を比べる（各データでの 重み×入力）----
out = pd.DataFrame({
    "元":   (X @ w0).numpy(),
    "RTN":  (X @ w1).numpy(),
    "GPTQ": (X @ w2).numpy(),
}, index=["データA", "データB", "データC", "データD"])
out["RTNのずれ"] = out["RTN"] - out["元"]
out["GPTQのずれ"] = out["GPTQ"] - out["元"]
display(out.round(3))

# ---- まとめ：重みのずれと出力のずれ（二乗の合計）----
summary = pd.DataFrame({
    "重みのずれ": [((w1 - w0)**2).sum().item(), ((w2 - w0)**2).sum().item()],
    "出力のずれ": [((X @ (w1 - w0))**2).sum().item(), ((X @ (w2 - w0))**2).sum().item()],
}, index=["RTN", "GPTQ"])
display(summary.round(3))

,元,RTN,GPTQ
x1,0.900000,0.600000,0.600000
x2,-0.600000,-0.600000,-0.600000
x3,-0.900000,-1.200000,-0.600000


,元,RTN,GPTQ,RTNのずれ,GPTQのずれ
データA,-1.200000,-1.800000,-1.200000,-0.600000,0.000000
データB,-0.600000,-1.800000,-0.600000,-1.200000,0.000000
データC,-0.600000,-0.600000,-0.600000,0.000000,0.000000
データD,-0.600000,-1.200000,-0.600000,-0.600000,0.000000


,重みのずれ,出力のずれ
RTN,0.180000,2.160000
GPTQ,0.180000,0.000000
